# 01 - Gaussian Process-based Bayesian Optimization

This notebook uses `GaussianProcessRegressor` as a **surrogate model** and builds a complete Bayesian Optimization loop for black-box minimization.

Terminology:

- `GaussianProcessRegressor`: probabilistic regression model,
- Bayesian Optimization: sequential optimization method,
- acquisition function: criterion used to choose the next expensive evaluation.

## 1. Algorithm

For minimization:

1. Evaluate several initial points.
2. Fit a GP surrogate to the observed data.
3. Predict \(\mu(x)\) and \(\sigma(x)\).
4. Compute an acquisition function.
5. Select the point with the most attractive acquisition score.
6. Evaluate the expensive true objective only at that point.
7. Add the observation and repeat until the evaluation budget is exhausted.

The objective is not to make the surrogate perfect everywhere. It is to use a limited evaluation budget efficiently.

## 2. Expected Improvement

Let \(f_{best}\) be the best observed objective value so far. For minimization, an improvement variable can be written as

\[I(x)=\max(f_{best}-f(x)-\xi,0).\]

Under the GP posterior, the Expected Improvement has the closed form

\[EI(x)=(f_{best}-\mu(x)-\xi)\Phi(z)+\sigma(x)\phi(z),\]

where

\[z=\frac{f_{best}-\mu(x)-\xi}{\sigma(x)}.\]

The parameter `xi` controls how strongly improvement beyond the current best is required and can influence exploration.

## 3. Lower Confidence Bound and direction

For minimization, a Lower Confidence Bound can be written as

\[LCB(x)=\mu(x)-\kappa\sigma(x).\]

LCB is naturally **minimized**. The implementation in this repository converts it to `score = -LCB` so every acquisition function follows a common convention: **larger score is better**.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        "Run this notebook from the repository root or from the notebooks directory."
    )

sys.path.insert(0, str((PROJECT_ROOT / "src").resolve()))
from gaussian_bo import GaussianProcessBayesOptimizer

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 4. Simple black-box example

Although the following function is available analytically, we deliberately treat it as a black box so we can test the optimizer against a known low-dimensional response.

In [ ]:
def objective_function(x):
    x0 = float(x[0])
    return (x0 - 2.0) ** 2 + 0.2 * np.sin(5.0 * x0)

optimizer = GaussianProcessBayesOptimizer(
    objective_function=objective_function,
    bounds=[[-2.0, 6.0]],
    n_initial_points=5,
    acquisition_function="ei",
    xi=0.01,
    random_state=42,
)

result = optimizer.optimize(n_iterations=15, verbose=True)
print("Best x:", result.best_x)
print("Best objective value:", result.best_y)

## 5. Convergence plot

The plot below shows the best observed objective value after each expensive function evaluation. A plateau means the optimizer has stopped finding better observations under the current budget; it does **not** prove that the global optimum has been found.

In [ ]:
best_so_far = np.minimum.accumulate(result.y_observed)

plt.figure(figsize=(9, 5))
plt.plot(np.arange(1, len(best_so_far) + 1), best_so_far, marker="o")
plt.xlabel("Number of true objective evaluations")
plt.ylabel("Best observed objective value")
plt.title("Bayesian Optimization convergence")
plt.grid(True)
plt.show()

## 6. Final GP surrogate

Because this example has one decision variable, we can visualize the final surrogate directly.

In [ ]:
X_grid = np.linspace(-2.0, 6.0, 500).reshape(-1, 1)
U_grid = optimizer._to_unit_space(X_grid)

optimizer.gp.fit(
    optimizer._to_unit_space(optimizer.X_observed),
    optimizer.y_observed,
)
mu, sigma = optimizer.gp.predict(U_grid, return_std=True)

plt.figure(figsize=(10, 5))
plt.plot(X_grid[:, 0], mu, label="GP predictive mean")
plt.fill_between(
    X_grid[:, 0],
    mu - 1.96 * sigma,
    mu + 1.96 * sigma,
    alpha=0.2,
    label="Approximate 95% uncertainty band",
)
plt.scatter(
    optimizer.X_observed[:, 0],
    optimizer.y_observed,
    label="True evaluations",
)
plt.xlabel("x")
plt.ylabel("Objective value")
plt.title("Final Gaussian Process surrogate")
plt.legend()
plt.grid(True)
plt.show()

## 7. Engineering interpretation

In a real industrial-engineering application, `objective_function` can be replaced by a simulation, physical test, digital twin, finite-element model, or another expensive evaluation pipeline. Bayesian Optimization is useful only when saving evaluations matters enough to justify fitting and optimizing the surrogate model.